In [8]:
# Project and symbolic tools
import heyoka as hy
import quivira as qv

# Numerics and visualization
import numpy as np
from _animation import animate_bar_model

In [9]:
model = qv.bar_joint_model()

# Define bars as pairs of endpoints
point_pairs = [
    ((0, 0), (1, 4)),
    ((1, 4), (0, 8)),
    ((0, 8), (4, 7)),
    ((4, 7), (8, 8)),
    ((8, 8), (9, 4)),
    ((9, 4), (8, 0)),
    ((8, 0), (4, -1)),
    ((4, -1), (0, 0)),
]

# Add all bars to the model
for start, end in point_pairs:
    model.add_bar_from_point_pair(start, end)

model.infer_joints()

model.add_support(0, -0.5, False, False)
#model.add_external_force(3, 0.5, 0, 50)
model.add_external_force(3, 0, 0, 0, -40)

ode_system, W = model.build_lagrange_eom(return_W=True)
ta = hy.taylor_adaptive(ode_system, compact_mode=True)

state0 = model.get_initial_state()
model.check_constraint_fulfillment(state0)

In [10]:
ta.time = 0.0
ta.state[:] = state0

# Compile the symbolic Delassus matrix W
W_array = np.asarray(W, dtype=object)
n = W_array.shape[0]
state_vars = [var for var, rhs in ode_system]

W_func = hy.cfunc(W_array.flatten().tolist(), state_vars, compact_mode=True)

# Stop when cond(W) exceeds 10^6
threshold = 1e-6

def get_rcond(state):
    W_num = np.asarray(W_func(state)).reshape(n, n)
    W_num = 0.5 * (W_num + W_num.T)

    if not np.all(np.isfinite(W_num)):
        return np.nan

    eigvals = np.linalg.eigvalsh(W_num)

    if eigvals[-1] <= 0:
        return 0.0

    return eigvals[0] / eigvals[-1]


def singularity_callback(ta):
    rcond = get_rcond(ta.state)

    if not np.isfinite(rcond) or rcond < threshold:
        print(f"Near-singularity detected at t = {ta.time:.8f}")
        print(f"rcond(W) = {rcond:.3e}")
        return False  # Stop integration

    return True  # Continue integration


# Second integration with singularity detection
t_end = 1.0
tgrid = np.linspace(0.0, t_end, 201)

if not np.isfinite(get_rcond(ta.state)) or get_rcond(ta.state) < threshold:
    raise ValueError("Second integration starts near a singularity")

result = ta.propagate_grid(
    tgrid,
    callback=singularity_callback,
    max_delta_t=0.001
)

solution = result[-1]
print(f"Integration ended at t = {ta.time:.8f}")


Integration ended at t = 1.00000000


In [11]:
# Use your existing 2D trajectory variable here:
animate_bar_model(model, solution)